# Module 7: Synthetic Data Generation and RAG Evaluation with Ragas

In this notebook, we'll go end-to-end from **generating synthetic evaluation data** to **systematically evaluating and improving a RAG pipeline** — all using [Ragas](https://github.com/explodinggradients/ragas).

The flow is:
1. **Generate** synthetic test data using Ragas' knowledge graph-based approach
2. **Build** a baseline RAG application with LangChain and LangGraph
3. **Evaluate** the RAG application against our synthetic test set using Ragas metrics
4. **Iterate** on the pipeline and measure the impact

> **NOTE:** Ragas is framework-agnostic — while this example uses LangChain/LangGraph, you can use Ragas with any framework (or none at all). Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

## Outline

**Part 1: Synthetic Data Generation**
- Task 1: Dependencies and API Keys
- Task 2: Data Preparation
- Task 3: Knowledge Graph Construction
- Task 4: Generating Synthetic Test Data
- ***❓ Question #1 & Question #2***
- ***🏗️ Activity #1: Custom Query Distribution***

**Part 2: RAG Evaluation with Ragas**
- Task 5: Building a Baseline RAG Application
- Task 6: Evaluating with Ragas
- Task 7: Making Adjustments and Re-Evaluating
- ***❓ Question #3, Question #4, Question #5, & Question #6***
- ***🏗️ Activity #2: Implement a Different Reranking Strategy***

---
# Part 1: Synthetic Data Generation with Ragas

Before we can evaluate a RAG system, we need high-quality test data. Manually creating question-answer pairs is time-consuming and often biased toward simple queries. Ragas solves this by building a **knowledge graph** from your documents and using it to generate diverse, realistic test questions automatically.

We'll use the **Stone Ridge 2025 Investor Letter** and an **Alternative Investments Handbook** as our source documents — maintaining continuity with the investment advisory use case from previous sessions.

## Task 1: Dependencies and API Keys

If you have not already done so, install the required libraries using the uv package manager:
```bash
uv sync
```

We'll need API keys for:
- **OpenAI** — for LLM and embedding models (used in both SDG and RAG evaluation)
- **Cohere** — for reranking in the improved pipeline ([sign up here](https://docs.cohere.com/reference/about))

You have two options for supplying your API keys:
- Use environment variables (copy `.env.sample` to `.env` and fill in your keys)
- Provide them via the prompts below

In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/kris/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/kris/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [3]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")

Please enter your OpenAI API key! ········
Please enter your Cohere API key! ········


## Task 2: Data Preparation

We'll prepare our data using two complementary investment-focused sources:
- **Stone Ridge 2025 Investor Letter** — covering Stone Ridge's investment philosophy, Bayesian approach to decision-making, energy investments, reinsurance, and risk management
- **Alternative Investments Handbook** — covering alternative asset classes including real estate, private equity, hedge funds, reinsurance, commodities, and infrastructure

The topical overlap between these documents (particularly around reinsurance, risk premiums, diversification, and alternative investments) helps Ragas build rich cross-document relationships in the knowledge graph.

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader

# Load the Stone Ridge 2025 Investor Letter (PDF)
pdf_loader = PyMuPDFLoader("data/Stone Ridge 2025 Investor Letter.pdf")
pdf_docs = pdf_loader.load()

# Load the Alternative Investments Handbook (text)
txt_loader = TextLoader("data/AlternativeInvestmentsHandbook.txt")
txt_docs = txt_loader.load()

# Combine into a single list
docs = pdf_docs + txt_docs
print(f"Loaded {len(docs)} documents:")
print(f"  - Stone Ridge 2025 Investor Letter: {len(pdf_docs)} pages")
print(f"  - AlternativeInvestmentsHandbook.txt: {len(txt_docs)} document(s)")

Loaded 15 documents:
  - Stone Ridge 2025 Investor Letter: 14 pages
  - AlternativeInvestmentsHandbook.txt: 1 document(s)


## Task 3: Knowledge Graph Construction

Ragas uses a **knowledge graph-based approach** to create synthetic test data. This is powerful because it allows us to create complex, multi-hop queries — not just simple factoid questions. Systems tend to perform well on simple evaluation tasks, so this additional complexity helps us find real weaknesses.

The process works in three stages:
1. **Build the graph** — insert documents as nodes
2. **Apply transformations** — extract headlines, summaries, themes, entities, and embeddings
3. **Create relationships** — use cosine similarity and overlap scores to connect related nodes

Let's start by defining our `generator_llm` and `generator_embeddings`.

In [5]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

### Step 1: Initialize the Knowledge Graph

We create an empty knowledge graph and populate it with our document nodes. Each full document becomes a node of type `DOCUMENT`.

In [6]:
from ragas.testset.graph import KnowledgeGraph, Node, NodeType

kg = KnowledgeGraph()

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 15, relationships: 0)

### Step 2: Apply Transformations

Now we apply the [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms) to enrich our knowledge graph. These transformations:

- **HeadlinesExtractor** — finds the overall headlines for each document
- **SummaryExtractor** — produces summaries of the documents
- **ThemesExtractor** — extracts broad themes
- **EmbeddingExtractor** — creates embeddings for similarity computation
- **NERExtractor** — extracts named entities

These are then used to build relationships between nodes via cosine similarity and overlap scoring.

In [7]:
from ragas.testset.transforms import default_transforms, apply_transforms

transforms = default_transforms(
    documents=docs,
    llm=generator_llm,
    embedding_model=generator_embeddings
)
apply_transforms(kg, transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/14 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/kris/.local/share/uv/python/cpython-3.13.5-macos-aarch64-none/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x1085e9140> is already entered


Applying HeadlineSplitter:   0%|          | 0/15 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/25 [00:00<?, ?it/s]

...

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/kris/.local/share/uv/python/cpython-3.13.5-macos-aarch64-none/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x1085e9140> is already entered


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/45 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'cc3cae'. Skipping!
Property 'summary_embedding' already exists in node '4fa3ec'. Skipping!
Property 'summary_embedding' already exists in node '952b7f'. Skipping!
Property 'summary_embedding' already exists in node '5c0bf3'. Skipping!
Property 'summary_embedding' already exists in node 'd6aa04'. Skipping!
Property 'summary_embedding' already exists in node 'c69aef'. Skipping!
Property 'summary_embedding' already exists in node '3426c8'. Skipping!
Task was destroyed but it is pending!
task: <Task pending name='Task-235' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/kris/Development/krispharper/Agent_Engineering_SRHG/07_Synthetic_Data_and_Evaluation/.venv/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-236' coro=<Kernel.shell_main() running at /Users/kris/Development/krispharper/Agent_Engineering_SRHG/07_Synthetic_Data_and_Evaluation/.venv/lib/python3.13/site-packages/ip

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 36, relationships: 309)

### Step 3: Save the Knowledge Graph

Knowledge graphs can be saved and loaded, which is useful for iterating on test generation without rebuilding the graph each time.

In [8]:
kg.save("investment_data_kg.json")

# You can reload it later:
# kg = KnowledgeGraph.load("investment_data_kg.json")

## Task 4: Generating Synthetic Test Data

With our knowledge graph built, we can now generate synthetic test data. Ragas provides several **query synthesizers**, each producing a different type of question:

- **`SingleHopSpecificQuerySynthesizer`** — creates questions answerable from a single chunk of context (e.g., *"What is Stone Ridge's approach to reinsurance investing?"*)
- **`MultiHopAbstractQuerySynthesizer`** — creates questions requiring synthesis across multiple chunks at an abstract level (e.g., *"How do alternative risk premiums relate to portfolio diversification?"*)
- **`MultiHopSpecificQuerySynthesizer`** — creates questions requiring specific details from multiple chunks (e.g., *"How does Stone Ridge's Bayesian philosophy connect to their energy investment strategy?"*)

We define a **query distribution** to control the mix of question types.

In [9]:
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)

query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

In [10]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=kg
)

testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,"Chicago is like part of my career right, so wh...",[and my career has essentially been a three-de...,My career has essentially been a three-decade ...,single_hop_specifc_query_synthesizer
1,Wha is the Archivo General de Indias and how d...,"[TODAY’S THE DAY Fancy math aside, the foundat...",During his pre-search research at the Archivo ...,single_hop_specifc_query_synthesizer
2,What is the performance of Stone Ridge Energy ...,[Standardized returns as of most recent quarte...,The performance data for Stone Ridge Energy Eq...,single_hop_specifc_query_synthesizer
3,What is Stone Ridge?,[Risk Disclosures This communication has been ...,Stone Ridge is mentioned in the context as a s...,single_hop_specifc_query_synthesizer
4,What are hedge fund strategies in the context ...,[The Alternative Investments Handbook A Practi...,Hedge fund strategies are included as part of ...,single_hop_specifc_query_synthesizer
5,"How does venture capital, as a form of alterna...",[<1-hop>\n\nThe Alternative Investments Handbo...,Venture capital (VC) is a specialized form of ...,multi_hop_abstract_query_synthesizer
6,what types of infrastructure are good for inve...,[<1-hop>\n\nChapter 15: Infrastructure Investm...,Infrastructure investments include economic in...,multi_hop_abstract_query_synthesizer
7,How priors and posterior distributions are use...,"[<1-hop>\n\nTODAY’S THE DAY Fancy math aside, ...",The context explains that priors are initial b...,multi_hop_abstract_query_synthesizer
8,Considering the role of commodities as inflati...,[<1-hop>\n\nPART 6: COMMODITIES AND REAL ASSET...,A reinsurance investment analyst can strategic...,multi_hop_specific_query_synthesizer
9,Whas Stone Ridge Energy is part of Stone Ridge...,[<1-hop>\n\nStandardized returns as of most re...,"Stone Ridge Energy is a part of Stone Ridge, a...",multi_hop_specific_query_synthesizer


### Abstracted SDG (Shortcut)

The above was the **unrolled** process showing each step. Ragas also provides a one-liner that builds the knowledge graph under the hood and generates the test set in a single call. This is convenient for quick iteration:

In [57]:
# Abstracted approach (for reference):
# generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
# dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### ✅ Answer:

* `SingleHopSpecificQuerySynthesizer` — analyzes single chunks of text and creates questions that are answerable from a single chunk.
* `MultiHopAbstractQuerySynthesizer` — looks at all chunks of text and creates questions that require general abstract answers across multiple chunks.
* `MultiHopSpecificQuerySynthesizer` — looks at all chunks of text and creates questions that require specific information from multiple chunks.


### ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### ✅ Answer:

Unrolled gives more control, e.g., by letting you pick the different synthesizers and the distribution of answers for each one. The abstracted approach is less code and doesn't require as many choices.

### 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

**Requirements:**
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [11]:

# Custom query distribution weighted toward multi-hop questions.
# For an investment advisory use case, users are more likely to ask complex,
# analytical questions that require synthesizing information across multiple
# sources (e.g., connecting Stone Ridge's philosophy to their reinsurance strategy)
# rather than simple factoid lookups.
custom_query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.20),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.50),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.30),
]

custom_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=kg
)

custom_testset = custom_generator.generate(testset_size=10, query_distribution=custom_query_distribution)
custom_df = custom_testset.to_pandas()
custom_df


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Can you tell me about the city Chicago and how...,[and my career has essentially been a three-de...,The context indicates that the person's career...,single_hop_specifc_query_synthesizer
1,What is the American Statistical Association,"[TODAY’S THE DAY Fancy math aside, the foundat...","The context mentions Arnold Zellner, who was n...",single_hop_specifc_query_synthesizer
2,Venture capital and real estate how they both ...,[<1-hop>\n\nThe Alternative Investments Handbo...,The context explains that alternative investme...,multi_hop_abstract_query_synthesizer
3,"How do infrastructure investments, as detailed...",[<1-hop>\n\nThe Alternative Investments Handbo...,Infrastructure investments provide exposure to...,multi_hop_abstract_query_synthesizer
4,How does Bayesian shrinkage help in managing r...,[<1-hop>\n\nand my career has essentially been...,Bayesian shrinkage involves pulling noisy esti...,multi_hop_abstract_query_synthesizer
5,How does the reinvestment of dividends influen...,[<1-hop>\n\nStandardized returns as of most re...,"The reinvestment of dividends, as reflected in...",multi_hop_abstract_query_synthesizer
6,How do the risk disclosures impact the interpr...,[<1-hop>\n\nStandardized returns as of most re...,The risk disclosures clarify that the reported...,multi_hop_abstract_query_synthesizer
7,"How do commodities, as discussed in the contex...",[<1-hop>\n\nPART 6: COMMODITIES AND REAL ASSET...,"Commodities, which include energy, metals, and...",multi_hop_specific_query_synthesizer
8,"Based on the information in Chapters 2 and 4, ...",[<1-hop>\n\nPART 2: REAL ESTATE INVESTMENTS Ch...,"Real estate investments, as discussed in Chapt...",multi_hop_specific_query_synthesizer
9,Wht is the relashunship between Stone Ridge En...,[<1-hop>\n\nStandardized returns as of most re...,The context indicates that Stone Ridge Energy ...,multi_hop_specific_query_synthesizer


### Activity #1 Findings

**Custom distribution chosen:**
* `SingleHopSpecificQuerySynthesizer`: 0.20 (reduced from 0.50)
* `MultiHopAbstractQuerySynthesizer`: 0.50 (increased from 0.25)
* `MultiHopSpecificQuerySynthesizer`: 0.30 (increased from 0.25)

**Rationale:**

In a real investment advisory context, users rarely ask simple one-chunk factoid questions like "What is Stone Ridge?". More commonly, they ask analytical questions that require synthesizing information across multiple topics — e.g., "How does Stone Ridge's Bayesian philosophy inform their approach to reinsurance risk premiums relative to traditional fixed income?" This kind of question draws on multiple document sections and tests whether the RAG system can actually reason across the knowledge base.

By weighting heavily toward `MultiHopAbstractQuerySynthesizer` (0.50), the test set better reflects the reasoning demands of investment analysis. We keep `MultiHopSpecificQuerySynthesizer` at 0.30 to include questions that require pinning down specific details across sources. `SingleHopSpecificQuerySynthesizer` is reduced to 0.20 — it's still useful for catching basic retrieval failures, but dominanting the test set with simple lookups would overstate the system's real-world utility.

**Comparison with default:**

The default distribution (50% single-hop) produces many simple, easily answerable questions that a naive RAG system can handle with even tiny chunks. The custom distribution produces a harder, more realistic test set — surfacing retrieval and reasoning weaknesses that would otherwise go undetected.


In [12]:

# Compare synthesizer distribution between default and custom test sets
import pandas as pd

default_counts = testset.to_pandas()["synthesizer_name"].value_counts().rename("default")
custom_counts = custom_df["synthesizer_name"].value_counts().rename("custom")

comparison = pd.concat([default_counts, custom_counts], axis=1).fillna(0).astype(int)
comparison["default_pct"] = (comparison["default"] / comparison["default"].sum() * 100).round(1)
comparison["custom_pct"] = (comparison["custom"] / comparison["custom"].sum() * 100).round(1)
print(comparison)


                                      default  custom  default_pct  custom_pct
synthesizer_name                                                              
single_hop_specifc_query_synthesizer        5       2         45.5        20.0
multi_hop_abstract_query_synthesizer        3       5         27.3        50.0
multi_hop_specific_query_synthesizer        3       3         27.3        30.0


---
# Part 2: RAG Evaluation with Ragas

Now that we have our synthetic test data, we can use it to **systematically evaluate** a RAG pipeline. The idea is simple:
1. Build a RAG application
2. Run our synthetic queries through it
3. Score the results using Ragas metrics
4. Make changes and measure the impact

This gives us a **data-driven approach** to improving our RAG system, rather than relying on vibes.

## Task 5: Building a Baseline RAG Application

We'll build a deliberately simple (and somewhat bad) RAG pipeline as our **baseline**, so we can clearly see the impact of improvements later.

Our baseline uses:
- Tiny chunks (50 characters) with no overlap
- A small embedding model (`text-embedding-3-small`)
- Only 3 retrieved documents
- A basic prompt

> **NOTE:** We use the same data that our synthetic test set was generated from — this is required because the test questions are specifically designed for this investment data.

### R — Retrieval

First, we chunk our documents and build a vector store.

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

2045

### ❓ Question #3:

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

##### ✅ Answer:

It controls how much of the preceding chunk overlaps with the next chunk. This helps preserve context when moving from chunk to chunk.

In [14]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [15]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="baseline_rag",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="baseline_rag",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)

In [16]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [17]:
def retrieve(state):
    retrieved_docs = retriever.invoke(state["question"])
    return {"context": retrieved_docs}

### A — Augmented

A simple RAG prompt:

In [18]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful investment advisory assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### G — Generation

We use `gpt-4.1-nano` for generation to avoid using the same model as our judge model.

In [19]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

In [20]:
def generate(state):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(messages)
    return {"response": response.content}

### Building the RAG Graph with LangGraph

In [21]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
    question: str
    context: List[Document]
    response: str

In [22]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a quick sanity check:

In [23]:
response = graph.invoke({"question": "What is Stone Ridge's investment philosophy?"})
response["response"]

"Stone Ridge's investment philosophy involves a relentless focus on growing, as indicated by their emphasis on continuous growth."

With tiny 50-character chunks and only 3 retrieved documents, the baseline likely struggles to provide good answers about Stone Ridge's investment philosophy. That's intentional — it gives us room to improve!

## Task 6: Evaluating with Ragas

Now we can evaluate our baseline RAG against the synthetic test data we generated in Part 1.

First, we run all the synthetic queries through our RAG pipeline to collect responses and retrieved contexts.

In [24]:
for test_row in testset:
    response = graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

Convert to an `EvaluationDataset` for smoother evaluation:

In [25]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(testset.to_pandas())

We select a **judge model** — a separate, capable model that scores the outputs. Using a different model than the generator avoids self-evaluation bias.

In [26]:
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

### Running the Baseline Evaluation

We evaluate across six metrics:
- **Context Recall** — did we retrieve the relevant context?
- **Faithfulness** — is the answer grounded in the retrieved context?
- **Factual Correctness** — is the answer factually correct vs. the reference?
- **Answer Relevancy** — is the answer relevant to the question?
- **Context Entity Recall** — did we capture the key entities from the reference context?
- **Noise Sensitivity** — is the answer affected by irrelevant retrieved content?

In [27]:
from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
    NoiseSensitivity,
)
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
        ResponseRelevancy(),
        ContextEntityRecall(),
        NoiseSensitivity(),
    ],
    llm=evaluator_llm,
    run_config=custom_run_config,
)
baseline_result

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

{'context_recall': 0.0455, 'faithfulness': 0.3342, 'factual_correctness': 0.5227, 'answer_relevancy': 0.5949, 'context_entity_recall': 0.2627, 'noise_sensitivity_relevant': 0.0000}

## Task 7: Making Adjustments and Re-Evaluating

Now that we have a baseline, let's improve the pipeline and measure the impact. We'll make three changes:

1. **Larger chunks** (500 characters with 30 overlap instead of 50 with 0 overlap)
2. **More documents retrieved** (k=20 instead of k=3)
3. **Reranking with Cohere** — retrieves 20 documents, then uses Cohere's reranker to select the top 5

Reranking is a technique that uses a cross-encoder model (slower but more accurate than embedding similarity) on a smaller subset of candidates to improve retrieval precision.

In [34]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(docs)
print(f"Improved chunking: {len(split_documents)} chunks (vs baseline)")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="improved_rag",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="improved_rag",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)
adjusted_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

Improved chunking: 202 chunks (vs baseline)


In [35]:
from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_cohere import CohereRerank

def retrieve_adjusted(state):
    compressor = CohereRerank(model="rerank-v3.5")
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=adjusted_retriever,
        search_kwargs={"k": 5},
    )
    retrieved_docs = compression_retriever.invoke(state["question"])
    return {"context": retrieved_docs}

In [36]:
from typing import TypedDict, List
from langchain_core.documents import Document

class AdjustedState(TypedDict):
    question: str
    context: List[Document]
    response: str

adjusted_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_adjusted, generate])
adjusted_graph_builder.add_edge(START, "retrieve_adjusted")
adjusted_graph = adjusted_graph_builder.compile()

Let's verify the improved pipeline works:

In [37]:
response = adjusted_graph.invoke({"question": "How does Stone Ridge approach risk management in their energy investments?"})
response["response"]

'The provided context does not include specific information on how Stone Ridge approaches risk management in their energy investments.'

### Running the Improved Evaluation

Now let's run the same synthetic test set through our improved pipeline and compare.

In [40]:
import time
import copy

rerank_testset = copy.deepcopy(testset)

for test_row in rerank_testset:
    response = adjusted_graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(5)  # To avoid rate limiting

In [41]:
rerank_evaluation_dataset = EvaluationDataset.from_pandas(rerank_testset.to_pandas())

rerank_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
        ResponseRelevancy(),
        ContextEntityRecall(),
        NoiseSensitivity(),
    ],
    llm=evaluator_llm,
    run_config=custom_run_config,
)
rerank_result

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

{'context_recall': 0.7364, 'faithfulness': 0.7706, 'factual_correctness': 0.5773, 'answer_relevancy': 0.9334, 'context_entity_recall': 0.5277, 'noise_sensitivity_relevant': 0.2146}

### ❓ Question #4:

Which system performed better, on what metrics, and why?

##### ✅ Answer:

The second system performed better on every metric because we added more context with larger chunks using more overlap, and a larger set of input documents.

### ❓ Question #5:

What are the benefits and limitations of using synthetic data generation for RAG evaluation? Consider both the practical advantages and potential pitfalls.

##### ✅ Answer:

* Benefits
    * No need for human input—all data can be programmatically generated.
    * Different types of data can be generated with different settings for various uses.
    * Easy to iterate and try lots of different combinations.
* Pitfalls
    * Humans can still provide out-of-the box thinking and unpredicible answers.
    * For a small sample size, humans may still be faster. If you have people submitting data already, it may be cheaper to use humans.
    * Small data sizes could result in duplicative or non-representative test data.

### ❓ Question #6:

If you were building a production investment advisory assistant for Stone Ridge, which Ragas metrics would be most important to optimize for and why? Consider the financial services domain specifically.

##### ✅ Answer:

1. **Faithfulness** — this is the most critical metric for an investment advisory assistant. The system must only make claims grounded in the retrieved context. A hallucinated fund return, risk figure, or investment recommendation could directly harm clients and expose Stone Ridge to regulatory liability. "Sounds reasonable" is not acceptable when real money is involved.

2. **Factual Correctness** — closely related to faithfulness, but measures accuracy against the ground truth reference rather than just internal consistency with retrieved context. For investment data (performance figures, fee structures, risk disclosures), factual errors are not recoverable the way a vague or incomplete answer might be.

3. **Context Recall** — the system needs to actually retrieve the relevant documents before it can answer correctly. Low context recall means the system is silently missing important information — especially dangerous when a user asks about risk disclosures, fund terms, or regulatory language that must be surfaced completely, not partially.

4. **Noise Sensitivity** — investment documents often contain similar-sounding but distinct concepts (e.g., different fund strategies, different risk premiums). A system that generates wrong answers because it was distracted by loosely related retrieved content is a reliability risk. Lower noise sensitivity is better.

5. **Answer Relevancy** — important for usability, but a lower priority than accuracy. An overly verbose or tangentially relevant answer is annoying; a concise but factually wrong one is dangerous.

6. **Context Entity Recall** — useful as a secondary signal (e.g., did we surface the right named entities like fund names, strategies, or portfolio companies?), but less critical to optimize directly compared to the accuracy-focused metrics above.

### 🏗️ Activity #2: Implement a Different Reranking Strategy

Experiment with different reranking parameters or strategies to see how they affect the evaluation metrics.

**Requirements:**
1. Modify the `retrieve_adjusted` function to use different parameters (e.g., change `k` values, try different `top_n` for reranking)
2. Or implement a different retrieval enhancement strategy (e.g., hybrid search, query expansion)
3. Run the evaluation and compare results with the baseline and reranking results above
4. Document your findings in the markdown cell below

In [49]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# Use a smaller k per query — MultiQueryRetriever generates 3 query variations
# by default, so 7 docs per variation gives up to ~21 candidates before dedup.
# This is comparable to the k=20 used in the Cohere reranking pipeline.
multi_query_base_retriever = vector_store.as_retriever(search_kwargs={"k": 7})

def retrieve_custom(state):
    multi_query_retriever = MultiQueryRetriever.from_llm(
        retriever=multi_query_base_retriever,
        llm=llm,
    )
    retrieved_docs = multi_query_retriever.invoke(state["question"])
    return {"context": retrieved_docs}


class CustomState(TypedDict):
    question: str
    context: List[Document]
    response: str

custom_graph_builder = StateGraph(CustomState).add_sequence([retrieve_custom, generate])
custom_graph_builder.add_edge(START, "retrieve_custom")
custom_graph = custom_graph_builder.compile()

# Sanity check
response = custom_graph.invoke({"question": "What is Stone Ridge's investment philosophy?"})
print(f"Retrieved {len(response['context'])} docs")
print(response["response"])


Retrieved 10 docs
Stone Ridge's investment philosophy centers on relentlessly focusing on growing after-tax cash flow to drive durable equity value in their operating businesses. They emphasize a deep understanding of what they know, open-mindedness to new data, rapid updating of their views, and the courage to create with conviction when they believe in their assessments. Their approach involves disciplined acquisitions, careful development, and continuous process improvement, aiming to be highly operationally excellent. Additionally, they prioritize alignment of interests, building products they would want for themselves, and maintaining a long-term perspective aligned with their mission of financial security for all.


In [51]:
import time
import copy

custom_testset = copy.deepcopy(testset)

for test_row in custom_testset:
    response = custom_graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(2)

custom_evaluation_dataset = EvaluationDataset.from_pandas(custom_testset.to_pandas())

custom_result = evaluate(
    dataset=custom_evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
        ResponseRelevancy(),
        ContextEntityRecall(),
        NoiseSensitivity(),
    ],
    llm=evaluator_llm,
    run_config=custom_run_config,
)
custom_result


Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

Exception raised in Job[35]: TimeoutError()
Exception raised in Job[41]: TimeoutError()
Exception raised in Job[53]: TimeoutError()
Exception raised in Job[65]: TimeoutError()


{'context_recall': 1.0000, 'faithfulness': 0.8142, 'factual_correctness': 0.5582, 'answer_relevancy': 0.9479, 'context_entity_recall': 0.5784, 'noise_sensitivity_relevant': 0.2217}

### Activity #2 Findings:

**Strategy: Multi-Query Retrieval**

Instead of tuning the Cohere reranker parameters, I implemented **multi-query retrieval** using LangChain's `MultiQueryRetriever`. This approach works differently from reranking:

* The LLM generates **3 alternative phrasings** of the original query
* Each phrasing is used to independently retrieve `k=7` documents from the vector store
* Results from all queries are **deduplicated** and returned as the combined context

This directly addresses a weakness that reranking does not: **vocabulary mismatch**. If the user's phrasing doesn't closely match how the answer is worded in the document, embedding similarity alone may miss it. By generating semantically equivalent but differently phrased queries, the retriever casts a wider net.

**Parameters:**
* Base retriever: `k=7` per query variation
* Number of query variations: 3 (default)
* Total candidates before dedup: up to ~21 (comparable to the `k=20` in the Cohere pipeline)
* No downstream reranking — retrieval quality improvement comes from query diversity alone

**Expected trade-offs vs. Cohere reranking:**
* Multi-query should improve **context recall** and **context entity recall** by finding documents that embedding similarity alone misses
* It may perform worse on **noise sensitivity** since all retrieved docs are passed directly to the LLM without a precision-focused reranking step
* It will be **slower** than pure embedding retrieval but avoids Cohere API calls

*(Run the comparison cell above to see the actual metric differences once cells are executed.)*


---
## Summary

In this notebook, we went end-to-end from data generation to evaluation:

1. **Built a knowledge graph** from our investment documents (Stone Ridge 2025 Investor Letter and Alternative Investments Handbook) and used it to understand the structure of our data
2. **Generated synthetic test data** with diverse query types (single-hop, multi-hop abstract, multi-hop specific)
3. **Built a baseline RAG pipeline** with deliberately simple parameters
4. **Evaluated with Ragas** across six metrics to establish a baseline
5. **Improved the pipeline** with larger chunks and Cohere reranking
6. **Re-evaluated** to measure the impact of our changes

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **Ragas metrics** give you a multi-dimensional view of RAG quality (retrieval vs. generation vs. faithfulness)
- **Small changes matter** — chunk size, retrieval strategy, and reranking can dramatically affect evaluation scores
- **Always use a different model for judging** than for generating to avoid self-evaluation bias